# 05 – RAG trên quy chế chi tiêu

**Mục tiêu:** truy xuất đúng điều quy chế để LLM giải thích và trích dẫn khi kiểm tra giao dịch.

**Thành phần**
| File | Vai trò |
|---|---|
| `data/policies/quy_che_chi_tieu.md` | Quy chế chi tiêu mẫu: 26 điều, 6 chương. Ngưỡng trùng với `config/rules.toml` |
| `src/rag.py` | Tách theo điều, BM25 (âm tiết + cặp âm tiết), embedding đa ngôn ngữ (fastembed, chạy trên máy), gộp bằng RRF |
| `data/policies/rag_eval.csv` | 50 câu hỏi tự soạn kèm điều đúng, diễn đạt khác lời văn quy chế |

**Truy xuất cho một hồ sơ giao dịch gồm hai phần:**
1. **Bắt buộc (logic):** điều gắn với rule đã vi phạm, với dấu hiệu bất thường và với nhóm chi phí, lấy theo metadata của điều. Phần này không phụ thuộc vào chất lượng tìm kiếm.
2. **Bổ sung (tìm kiếm):** tìm kiếm kết hợp trên câu truy vấn sinh tự động từ hồ sơ, để lấy thêm điều về ngữ cảnh.

Groq không có API embedding, nên embedding chạy trên máy bằng `fastembed` (ONNX, không cần GPU hay PyTorch).

In [1]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.rag import parse_policy, PolicyRetriever, tokenize

pd.set_option('display.width', 180); pd.set_option('display.max_colwidth', 90)
BLUE, ORANGE, INK, INK2, GRID = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({
    'font.family': ['Arial', 'DejaVu Sans'], 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': INK2, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'xtick.color': INK2, 'ytick.color': INK2,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'figure.dpi': 110, 'figure.facecolor': 'white',
})

## 1. Quy chế sau khi tách theo điều

In [2]:
chunks = parse_policy()
pd.DataFrame([{'Mã': c.id, 'Điều': c.title, 'Chương': c.chapter, 'Rule': ','.join(c.rules),
               'Nhóm chi phí': ','.join(c.groups), 'Tín hiệu': ','.join(c.signals),
               'Số âm tiết': len(tokenize(c.text)) // 2} for c in chunks]).set_index('Mã')

,Điều,Chương,Rule,Nhóm chi phí,Tín hiệu,Số âm tiết
Mã,,,,,,
D01,Điều 1. Phạm vi và đối tượng áp dụng,Chương I. Quy định chung,,,,87
D02,Điều 2. Nguyên tắc chi tiêu,Chương I. Quy định chung,,,,100
D03,Điều 3. Trách nhiệm của người chi,Chương I. Quy định chung,"R01,R03",,,67
D04,Điều 4. Hạn mức một giao dịch,Chương II. Hạn mức và phê duyệt,R02,,,81
D05,Điều 5. Nghiêm cấm chia nhỏ giao dịch,Chương II. Hạn mức và phê duyệt,R04,,burst,121
D06,Điều 6. Thẩm quyền phê duyệt,Chương II. Hạn mức và phê duyệt,"R02,R05",,,65
D07,Điều 7. Chứng từ bắt buộc,Chương III. Chứng từ và ghi nhận,R01,,,89
D08,Điều 8. Hóa đơn trùng,Chương III. Chứng từ và ghi nhận,R03,,,125
D09,Điều 9. Thời hạn ghi nhận giao dịch,Chương III. Chứng từ và ghi nhận,,,late_post,91


## 2. Đánh giá truy xuất trên 50 câu hỏi

In [3]:
ev = pd.read_csv(ROOT / 'data' / 'policies' / 'rag_eval.csv')
ev['rel'] = ev.relevant.str.split('|')
print(f'{len(ev)} câu hỏi; {ev.rel.map(len).gt(1).sum()} câu có từ 2 điều đúng trở lên')
ev.head(5)

50 câu hỏi; 9 câu có từ 2 điều đúng trở lên


,question,relevant,rel
0,Mỗi lần quẹt thẻ tối đa được bao nhiêu tiền?,D04,[D04]
1,Mua máy chủ giá 12 nghìn đô thì có dùng thẻ được không?,D04|D21,"[D04, D21]"
2,Nhân viên tách đơn hàng thành ba lần thanh toán để không phải xin duyệt thì xử lý sao?,D05,[D05]
3,Hai lần mua ở cùng một nơi trong hai ngày mỗi lần 3000 đô có vấn đề gì không?,D05,[D05]
4,Ai có quyền duyệt khoản chi vượt ngân sách?,D06|D11,"[D06, D11]"


In [4]:
def metrics(rank_fn):
    rows = []
    for q, rel in zip(ev.question, ev.rel):
        ranking = rank_fn(q)
        pos = [ranking.index(x) for x in rel]
        rows.append({'Hit@1': min(pos) < 1, 'Recall@3': np.mean([p < 3 for p in pos]),
                     'Recall@5': np.mean([p < 5 for p in pos]), 'MRR': 1 / (min(pos) + 1)})
    return pd.DataFrame(rows).mean()

models = {'MiniLM-L12 (0,22 GB)': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
          'mpnet-base (1,0 GB)': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'}
results, retrievers = {}, {}
for name, m in models.items():
    t0 = time.time()
    r = retrievers[name] = PolicyRetriever(chunks, embed_model=m)
    print(f'{name}: nạp model + lập chỉ mục {time.time() - t0:.1f} giây')
    if not results:
        results['BM25 (âm tiết + cặp âm tiết)'] = metrics(r.rank_bm25)
    results[f'Embedding – {name}'] = metrics(r.rank_dense)
    results[f'Kết hợp RRF – {name}'] = metrics(r.rank_hybrid)

# BM25 chỉ dùng âm tiết đơn, để thấy tác dụng của cặp âm tiết
from rank_bm25 import BM25Okapi
import re
uni = BM25Okapi([re.findall(r'\w+', c.full_text.lower()) for c in chunks])
results['BM25 (chỉ âm tiết đơn)'] = metrics(lambda q: [chunks[i].id for i in np.argsort(-uni.get_scores(re.findall(r'\w+', q.lower())))])

res = pd.DataFrame(results).T.loc[[
    'BM25 (chỉ âm tiết đơn)', 'BM25 (âm tiết + cặp âm tiết)',
    'Embedding – MiniLM-L12 (0,22 GB)', 'Embedding – mpnet-base (1,0 GB)',
    'Kết hợp RRF – MiniLM-L12 (0,22 GB)', 'Kết hợp RRF – mpnet-base (1,0 GB)']]
res.style.format('{:.3f}').highlight_max(color='#dbe8f8')

MiniLM-L12 (0,22 GB): nạp model + lập chỉ mục 1.8 giây


mpnet-base (1,0 GB): nạp model + lập chỉ mục 5.3 giây


,Hit@1,Recall@3,Recall@5,MRR
BM25 (chỉ âm tiết đơn),0.720,0.900,0.930,0.819
BM25 (âm tiết + cặp âm tiết),0.800,0.870,0.970,0.864
"Embedding – MiniLM-L12 (0,22 GB)",0.620,0.720,0.840,0.740
"Embedding – mpnet-base (1,0 GB)",0.700,0.810,0.890,0.797
"Kết hợp RRF – MiniLM-L12 (0,22 GB)",0.740,0.910,0.970,0.844
"Kết hợp RRF – mpnet-base (1,0 GB)",0.720,0.910,0.950,0.835


**Nhận xét:**
- **BM25 rất mạnh** trên bộ quy chế này. Các câu hỏi về quy chế thường chứa đúng thuật ngữ nghiệp vụ ("hạn mức", "hóa đơn", "ngân sách"), và tài liệu chỉ có 26 điều.
- **Thêm cặp âm tiết** giúp BM25 nhận ra từ ghép tiếng Việt ("hóa đơn", "khách sạn"), cải thiện rõ so với chỉ dùng âm tiết đơn.
- **Embedding riêng lẻ kém BM25**, nhưng tìm được các câu hỏi diễn đạt khác hẳn quy chế. Khi **kết hợp RRF**, Recall@3 và Recall@5 cao nhất. Trong RAG, LLM đọc nhiều điều cùng lúc, nên việc điều đúng nằm trong top-k quan trọng hơn việc nó đứng đầu. Vì vậy hệ thống dùng **kết hợp RRF với MiniLM**: model nhỏ hơn 4 lần mà kết quả tương đương.
- **Hạn chế:** tập đánh giá nhỏ (50 câu) và do chính người xây dựng hệ thống soạn, nên số liệu có độ dao động lớn. Cần bổ sung câu hỏi từ người dùng thật.

In [5]:
r = retrievers['MiniLM-L12 (0,22 GB)']
miss = []
for q, rel in zip(ev.question, ev.rel):
    top3 = r.rank_hybrid(q)[:3]
    if not set(rel) & set(top3):
        miss.append({'Câu hỏi': q, 'Điều đúng': ','.join(rel), 'Top 3 truy xuất': ','.join(top3)})
print(f'Số câu hỏi không có điều đúng nào trong top 3: {len(miss)}/{len(ev)}')
pd.DataFrame(miss)

Số câu hỏi không có điều đúng nào trong top 3: 3/50


,Câu hỏi,Điều đúng,Top 3 truy xuất
0,Đổ xăng cho xe cá nhân có được dùng thẻ không?,D19,"D14,D01,D03"
1,Hệ thống gợi ý sai loại chi phí thì ai sửa?,D23,"D02,D11,D24"
2,Số tiền cao gấp nhiều lần mức chi thường ngày của người chi nghĩa là gì?,D24,"D08,D26,D12"


## 3. Truy xuất cho hồ sơ giao dịch

Với một giao dịch thật, câu truy vấn được sinh tự động từ hồ sơ: nhóm chi phí, mô tả mặt hàng, thông báo vi phạm rule, lý do bất thường. Các điều gắn với rule và dấu hiệu đã phát hiện được lấy **bắt buộc**.

In [6]:
from src.case import CaseEngine
t0 = time.time()
engine = CaseEngine()
print(f'Nạp dữ liệu và mô hình: {time.time() - t0:.0f} giây')
df = engine.df
examples = {
    'Nhiều vi phạm rule': engine.hits.groupby('txn_id').size().sort_values().index[-1],
    'Chỉ mô hình bất thường': df[engine.tx_flag & (df.rule_level == 'Bình thường')].txn_id.iloc[-1],
    'Cụm mua dồn': df[df.burst_day_flag & (df.month >= pd.Period('2026-06', 'M'))].txn_id.iloc[0],
}
rows = []
for kind, txn in examples.items():
    c = engine.case(txn)
    rules = [h['rule_id'] for h in c['rule_hits']]
    got = r.retrieve_for_case(engine.retrieval_query(c), rule_ids=rules, signals=c['anomaly']['signals'],
                              groups=[c['expense_group']['group']])
    required = {x.id for x in r.required_for(rules, c['anomaly']['signals'], [c['expense_group']['group']])}
    rows.append({'Loại': kind, 'Giao dịch': txn, 'Rule': ','.join(rules), 'Tín hiệu': ','.join(c['anomaly']['signals']),
                 'Điều bắt buộc': ','.join(x.id for x in got if x.id in required),
                 'Điều bổ sung (tìm kiếm)': ','.join(x.id for x in got if x.id not in required)})
pd.DataFrame(rows)

Nạp dữ liệu và mô hình: 33 giây


,Loại,Giao dịch,Rule,Tín hiệu,Điều bắt buộc,Điều bổ sung (tìm kiếm)
0,Nhiều vi phạm rule,202605-010177,"R01,R02,R03,R05,R07",,"D03,D04,D06,D07,D08,D11,D13,D19","D05,D10"
1,Chỉ mô hình bất thường,202608-035663,R09,"amount,new_group,new_merchant","D16,D15,D18,D13,D24","D08,D17"
2,Cụm mua dồn,202606-000299,,burst,"D17,D05,D26,D24","D08,D04"


In [7]:
print(engine.retrieval_query(engine.case(examples['Chỉ mô hình bất thường'])))

Lưu trú khách sạn
ROOM CHARGES
Giao dịch ngày nghỉ: Giao dịch ngày 22/08/2026 (Thứ Bảy), ngoài ngày làm việc.
Lần đầu người này chi cho nhóm chi phí này
Số tiền $724.50 gấp 3.8 lần mức chi thường gặp tại cửa hàng này
Lần đầu người này mua ở cửa hàng này


## 4. Kết luận

1. Quy chế mẫu gồm 26 điều có metadata (rule, nhóm chi phí, tín hiệu). Nhờ vậy các điều liên quan tới vi phạm được lấy **theo logic**, không phụ thuộc vào chất lượng tìm kiếm.
2. Tìm kiếm kết hợp (BM25 có cặp âm tiết + embedding MiniLM, gộp RRF) cho Recall@3 và Recall@5 cao nhất trên 50 câu hỏi. Embedding chạy trên máy, không cần GPU.
3. **Bước tiếp theo:** đưa hồ sơ + điều đã truy xuất cho LLM (Groq) để sinh giải thích có trích dẫn (`src/llm.py`), và kiểm chứng trích dẫn tự động.